In [1]:
! pip install psycopg2-binary pandas


In [ ]:
import psycopg2
import pandas as pd

# Connect to PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="....",
    host="localhost",
    port="5432"
)

# Define your SQL query
query = """
SELECT
    c.PID,
    -- Extract the Year from DateCompleted, ArticleDates, or History
    COALESCE(
        (
            -- Extract Year from DateCompleted after fixing format
            REPLACE(
                REPLACE(c.data->>'DateCompleted', '''', '"'),
                '""', '"'
            )::jsonb->>'Year'
        ),
        (
            -- Extract Year from the last element of ArticleDates if DateCompleted is null
            (
                SELECT 
                    (jsonb_array_elements(c.data->'ArticleDates')->>'Year')
                FROM jsonb_array_elements(c.data->'ArticleDates') AS x
                ORDER BY (x->>'Year')::int DESC
                LIMIT 1
            )
        ),
        (
            -- Extract Year from the last element of History if both DateCompleted and ArticleDates are null
            (
                SELECT 
                    (jsonb_array_elements(c.data->'History')->>'Year')
                FROM jsonb_array_elements(c.data->'History') AS x
                ORDER BY (x->>'Year')::int DESC
                LIMIT 1
            )
        )
    ) AS PublishedDate,

    c.data->>'ArticleTitle' AS Title,
    c.data->>'Abstract' AS Abstract,
    c.*
FROM public.canada AS c
LEFT JOIN public.m1 AS m1 ON c.pid = m1.pid
LEFT JOIN public.m2 AS m2 ON m1.pid = m2.pid
"""

# Load data into a DataFrame
df = pd.read_sql_query(query, conn)

# Close the connection
conn.close()

# Display the DataFrame
print(df.head())


  ?column? ?column?       pid
0    false    false  28444633


/var/folders/gx/4hmfp0zd12bblv6znjwbmv600000gn/T/ipykernel_81032/3259150840.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
